<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/rag_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RAG sequence

In [ ]:
!pip -q install transformers "datasets<3" sentencepiece accelerate faiss-cpu

In [ ]:
import re
import torch
import transformers
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

torch.manual_seed(0)

# Load tokenizer and retriever

In [ ]:
tokenizer = RagTokenizer.from_pretrained("facebook/rag-sequence-nq")

In [ ]:
retriever = RagRetriever.from_pretrained("facebook/rag-sequence-nq", index_name = "compressed")

In [ ]:
model = RagSequenceForGeneration.from_pretrained("facebook/rag-sequence-nq", retriever = retriever)
model = model.to(device)

# Test Question


In [ ]:
question = "What is the capital of France?"

inputs = tokenizer(question, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
  generated_ids = model.generate(**inputs)

answer = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Question: ", question)
print("Generated answer: ", answer)

# Connecting NQ

In [ ]:
nq = load_dataset("sentence-transformers/natural-questions", split="train")

In [ ]:
def get_question(data_line):
  return data_line['query']

def get_answer(data_line):
  answer = data_line['answer']
  if isinstance(answer, list):
    answer = answer[0]
  return answer

# Test NQ question

In [ ]:
nq_question = nq[0]
question = get_question(nq_question)
actual_answer = get_answer(nq_question)

inputs = tokenizer(question, return_tensors = "pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
  generated_ids = model.generate(**inputs)

answer = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Question: ", question)
print("Actual answer: ", actual_answer)
print("Generated answer: ", answer)

# Test 10 NQ questions

In [ ]:
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
results = []

for i in range(10):
  nq_question = nq[i]
  question = get_question(nq_question)
  answer = get_answer(nq_question)


  inputs = tokenizer(question, return_tensors = "pt")
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    generated_ids = model.generate(**inputs)

  prediction = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

  answer_normalized = normalize_text(answer)
  prediction_normalized = normalize_text(prediction)

  em = (int)(answer_normalized==prediction_normalized)

  results.append({
      "question_number": i,
      "question": question,
      "answer": answer,
      "prediction": prediction,
      "em": em
  })

df_rag = pd.DataFrame(results)
df_rag.head()

# Accuracy

In [ ]:
accuracy = df_rag['em'].mean() * 100
print("RAG Accuracy: ", accuracy)

# Saving Results

In [ ]:
df_rag.to_csv('rag_sequence_results.csv', index=False)
print("Results saved to rag_sequence_results.csv")